## 4) Data

In [253]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

import warnings
warnings.filterwarnings("ignore")

In [349]:
# Full dataset: 1 million+ observations and 24 columns
data_full = pd.read_csv('movies.csv')
data_full.shape

(1375026, 24)

In [350]:
# Adding profit column
data_full['profit'] = data_full['revenue'] - data_full['budget']

# Filtering out observations with missing values in any of the predictors I want to use: 
data_clean = data_full.loc[~(data_full['popularity'].isna() |
               data_full['genres'].isna() |
               data_full['runtime'].isna() |
               data_full['release_date'].isna() |
               data_full['adult'].isna() |
               data_full['original_language'].isna())]

data_clean = data_clean.loc[~((data_clean['revenue'] == 0) & (data_clean['budget'] == 0))]

In [ ]:
# Table with missing values in the predictors of interest
data_full[['popularity', 'genres', 'runtime', 'original_language', 'release_date', 'adult']].isna().sum()

popularity                0
genres               597001
runtime                   0
original_language         0
release_date         293527
adult                     0
dtype: int64

In [ ]:
# Observations with unrealistic runtimes
print('Observation with extremely high runtime:')
display(data_clean.loc[data_clean['runtime'] == data_clean['runtime'].max(), ['title', 'runtime']])

print(' ')

print('Observation(s) with extremely low runtimes:')
display(data_clean.loc[data_clean['runtime'] == data_clean['runtime'].min(), ['title', 'runtime']].head(3)) 

Observation with extremely high runtime:


,title,runtime
964483,The Innocence,1265


 
Observation(s) with extremely low runtimes:


,title,runtime
18758,Vajont,0
23015,Foxter and Max,0
24920,Il regno,0


In [353]:
# Observations where runtime is greater than 30 minutes and less than 300 minutes (5 hours)
data_clean = data_clean.loc[(data_clean['runtime'] >= 30) & (data_clean['runtime'] <= 300)]

In [354]:
# Encoding the "genres" column so that it is no longer a long, singular list with many unique combinations
mlb_genres = MultiLabelBinarizer()
genres_split = data_clean['genres'].str.split(', ') # using the comma , to split genres
genres_encoded = mlb_genres.fit_transform(genres_split)
genres_df = pd.DataFrame(genres_encoded, columns = mlb_genres.classes_, index = data_clean.index)

# Collapse rare genres into 'Other' (appear in less than 1% of films)
threshold = 0.01
genre_counts = genres_df.sum()
common_genres = genre_counts[genre_counts / len(genres_df) >= threshold].index
rare_genres = genre_counts[genre_counts / len(genres_df) < threshold].index

genres_df['Other_Genre'] = (genres_df[rare_genres].sum(axis = 1) > 0)
genres_df = genres_df[list(common_genres) + ['Other_Genre']]

# Drop original and join encoded genres
data_clean = data_clean.drop(columns = ['genres'])
data_clean = data_clean.join(genres_df)

In [355]:
# Cleaning original_language variable:
    # For languages that appear in less than 1% of movies, this will be labeled as "Other".
min_pct = 0.01
lang_counts = data_clean['original_language'].value_counts()
top_orig_langs = lang_counts[lang_counts / len(data_clean) >= min_pct].index.tolist()

data_clean['original_language'] = data_clean['original_language'].apply(
    lambda x: x if x in top_orig_langs else 'other')

In [ ]:
# original_language table showing the languages that were kept with their respective counts and percentages

lang_counts = data_clean['original_language'].value_counts()
lang_pct = (lang_counts / len(data_clean) * 100).round(2)

lang_table = pd.DataFrame({
    'Language': lang_counts.index,
    'Count': lang_counts.values,
    'Percentage': lang_pct.values
}).sort_values('Percentage', ascending = False).reset_index(drop=True)

lang_table['Percentage'] = lang_table['Percentage'].apply(lambda x: f"{x}%")

lang_table.style.hide(axis = 'index')

Language,Count,Percentage
en,23441,69.92%
other,3978,11.87%
fr,1252,3.73%
es,1143,3.41%
ru,687,2.05%
ja,622,1.86%
hi,564,1.68%
de,547,1.63%
zh,477,1.42%
it,459,1.37%


In [357]:
# Extracting month only from the release_date column
data_clean['release_date'] = pd.to_datetime(data_clean['release_date'])
data_clean['release_month'] = data_clean['release_date'].dt.month

data_clean = data_clean.drop(columns = ['release_date']) # Removing original release_date column

In [358]:
# Randomly sampling (without replacement) 25,000 observations due to computational practicality
data = data_clean.sample(n = 25000, random_state = 16) # random_state for reproducibility

In [ ]:
# Dataframe showing the number of observations after obtaining a random sample and cleaning the data
pd.DataFrame([{'Number of Observations': data.shape[0],
             'Number of Columns': data.shape[1]}]).style.hide(axis = 'index')

Number of Observations,Number of Columns
25000,44


In [335]:
# Splitting data into training and test sets (80-20 split)
genre_cols = [col for col in data.columns if col in list(common_genres) + ['Other_Genre']]
feature_cols = ['popularity', 'runtime', 'original_language', 'release_month', 'adult'] + genre_cols

y = data['profit']
X = data[feature_cols] # So I don't have to manually list out each genre column

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 16)



# One-hot-encoding original_language and release_month
    # original_language
X_train = pd.get_dummies(X_train, columns = ['original_language'], drop_first = True)
X_test = pd.get_dummies(X_test, columns = ['original_language'], drop_first = True)
X_test = X_test.reindex(columns = X_train.columns) # Align columns in case some categories only appear in one split

    # release_month
X_train = pd.get_dummies(X_train, columns = ['release_month'], drop_first = True)
X_test = pd.get_dummies(X_test, columns = ['release_month'], drop_first = True)
X_test = X_test.reindex(columns = X_train.columns) # Align columns in case some categories only appear in one split

## 5) Prediction

#### Unregularized/Linear Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# Model with 1 predictor (popularity)
model = LinearRegression()
model.fit(X_train[['popularity']], y_train)

y_pred_train = model.predict(X_train[['popularity']])
y_pred_test = model.predict(X_test[['popularity']])


# Training/test performances as a dataframe
pd.DataFrame([{
    'Predictors': 'popularity',
    'Train RMSE': round(root_mean_squared_error(y_train, y_pred_train), 2),
    'Test RMSE': round(root_mean_squared_error(y_test, y_pred_test), 2),
    'Train MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2),
    'Train R²': round(model.score(X_train[['popularity']], y_train), 4),
    'Test R²': round(model.score(X_test[['popularity']], y_test), 4)}]).style.hide(axis = 'index') # hides row index for cleaner visual in report

Predictors,Train RMSE,Test RMSE,Train MAE,Test MAE,Train R²,Test R²
popularity,85513064.020000,74852384.120000,24842043.400000,23697003.660000,0.057200,0.005600


In [ ]:
# Adding the remaining predictors in the linear/unregularized model

genre_cols_clean = [col.replace(' ', '_') for col in genre_cols] # Replace spaces with underscores within genres to prevent errors in model formulas
lang_cols = [col for col in X_train.columns if col.startswith('original_language_')] # Grabbing all ohe language columns (instead of manually listing)
month_cols = [col for col in X_train.columns if col.startswith('release_month_')] # Grabbing all ohe month columns (instead of manually listing)

# Rename columns with spaces in X_train and X_test: leaving spaces causes KeyErrors
X_train = X_train.rename(columns = {'Science Fiction': 'Science_Fiction', 'TV Movie': 'TV_Movie'})
X_test = X_test.rename(columns = {'Science Fiction': 'Science_Fiction', 'TV Movie': 'TV_Movie'})



# Ordered dictionary where each key is a label for the table and each value is a cumulative list of predictors at each step
    # Each step builds on the previous one
predictor_steps = {
    'popularity': ['popularity'],
    'popularity, genres': ['popularity'] + genre_cols_clean,
    'popularity, genres, release_month': ['popularity'] + genre_cols_clean + month_cols,
    'popularity, genres, release_month, runtime': ['popularity'] + genre_cols_clean + month_cols + ['runtime'],
    'popularity, genres, release_month, runtime, original_language': ['popularity'] + genre_cols_clean + month_cols + ['runtime'] + lang_cols,
    'popularity, genres, release_month, runtime, original_language, adult': ['popularity'] + genre_cols_clean + month_cols + ['runtime'] + lang_cols + ['adult']}

results = []

# Looping through each entry in predictor_steps, creating a linear unregularized model on the current set of predictors 
    # and adding training/test performance metrics for comparison
for label, predictors in predictor_steps.items():
    model = LinearRegression()
    model.fit(X_train[predictors], y_train)
    
    y_pred_train = model.predict(X_train[predictors])
    y_pred_test = model.predict(X_test[predictors])
    
    results.append({
        'Predictors Added': label,
        'Train RMSE': round(root_mean_squared_error(y_train, y_pred_train), 2),
        'Test RMSE': round(root_mean_squared_error(y_test, y_pred_test), 2),
        'Train MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
        'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2),
        'Train R²': round(model.score(X_train[predictors], y_train), 4),
        'Test R²': round(model.score(X_test[predictors], y_test), 4)})

# Training and test perofrmances of all predictors
pd.DataFrame(results).style.hide(axis = 'index').set_properties(subset = ['Predictors Added'], **{'text-align': 'left'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]},
    {'selector': 'th.col0', 'props': [('text-align', 'left')]}])

Predictors Added,Train RMSE,Test RMSE,Train MAE,Test MAE,Train R²,Test R²
popularity,85513064.020000,74852384.120000,24842043.400000,23697003.660000,0.057200,0.005600
"popularity, genres",83845230.430000,73910802.290000,24558881.960000,23690161.340000,0.093600,0.030500
"popularity, genres, release_month",83685284.870000,73688898.140000,25512537.080000,24531277.790000,0.097000,0.036300
"popularity, genres, release_month, runtime",83121514.350000,73176915.760000,27182909.700000,26418285.870000,0.109200,0.049700
"popularity, genres, release_month, runtime, original_language",82724253.450000,72823171.050000,28036541.680000,27412113.450000,0.117700,0.058800
"popularity, genres, release_month, runtime, original_language, adult",82721748.810000,72818575.120000,28042236.000000,27414291.840000,0.117700,0.058900


#### Non-linear/Regularized Model

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LassoCV

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Degree 2 polynomials and interaction terms
poly = PolynomialFeatures(degree = 2, include_bias = False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)


# LassoCV to find best alpha
lcv = LassoCV(cv = 5, random_state = 16, max_iter = 10000)
lcv.fit(X_train_poly, y_train)

print(f"Best alpha: {lcv.alpha_}")

Best alpha: 3429325.392696592


In [ ]:
# Table with all terms with nonzero coefficients, ordered by magnitude
coefficients[coefficients['Coefficient'] != 0].reindex(
    coefficients[coefficients['Coefficient'] != 0]['Coefficient'].abs().sort_values(ascending = False).index).style.hide(axis = 'index') 


Term,Coefficient
popularity,23689514.984142
popularity runtime,16867826.488874
popularity Adventure,7743083.417891
runtime,6663376.673534
popularity original_language_en,5476992.948019
runtime Adventure,3881332.874327
runtime Science_Fiction,3578165.138943
popularity release_month_9,-3555560.096921
popularity release_month_12,3477134.061116
popularity Fantasy,3139990.490508


In [346]:
# Three most important non-linear terms
poly_feature_names = poly.get_feature_names_out(X_train.columns) # Feature names

coefficients = pd.DataFrame({'Term': poly_feature_names, 'Coefficient': lcv.coef_}) # coefficients of each term

top_3 = coefficients[coefficients['Coefficient'] != 0]  # remove terms Lasso zeroed out
top_3 = top_3.reindex(top_3['Coefficient'].abs().sort_values(ascending = False).index)
top_3 = top_3.head(3).reset_index(drop = True)

top_3.style.hide(axis = 'index')

Term,Coefficient
popularity,23689514.984142
popularity runtime,16867826.488874
popularity Adventure,7743083.417891


In [347]:
# Evaluate on test set
y_pred_test = lcv.predict(X_test_poly)
y_pred_train = lcv.predict(X_train_poly)

# Dataframe of training and test performances
pd.DataFrame([{
    'Training RMSE': round(root_mean_squared_error(y_train, y_pred_train), 2),
    'Test RMSE': round(root_mean_squared_error(y_test, y_pred_test), 2),
    'Training MAE': round(mean_absolute_error(y_train, y_pred_train), 2),
    'Test MAE': round(mean_absolute_error(y_test, y_pred_test), 2),
    'Training R²': round(lcv.score(X_train_poly, y_train), 4),
    'Test R²': round(lcv.score(X_test_poly, y_test), 4)}]).style.hide(axis = 'index')

Training RMSE,Test RMSE,Training MAE,Test MAE,Training R²,Test R²
75450088.250000,69520429.820000,21311488.320000,20734819.930000,0.266000,0.142300


## 6) Inference

In [348]:
# statsmodels model and summary
import statsmodels.formula.api as smf

# Replace spaces with underscores in columns to avoid errors in statsmodels formula
inference_df = X_train.copy()
inference_df['profit'] = y_train.values
inference_df = inference_df.rename(columns = {'Science Fiction': 'Science_Fiction', 'TV Movie': 'TV_Movie'})

# Using : instead of * for interaction terms to avoid redundantly adding predictors again
formula = '''profit ~ popularity + runtime + Adventure + Science_Fiction + original_language_en + release_month_12 + 
I(popularity**2) + I(runtime**2) + popularity:runtime + popularity:Adventure + runtime:Adventure + runtime:Science_Fiction + 
popularity:original_language_en + popularity:release_month_12'''

inference_model = smf.ols(formula = formula, data = inference_df).fit()
inference_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 profit   R-squared:                       0.252
Model:                            OLS   Adj. R-squared:                  0.252
Method:                 Least Squares   F-statistic:                     482.0
Date:                Sun, 15 Mar 2026   Prob (F-statistic):               0.00
Time:                        12:28:18   Log-Likelihood:            -3.9134e+05
No. Observations:               20000   AIC:                         7.827e+05
Df Residuals:                   19985   BIC:                         7.828e+05
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
===========================================================================================================
                                              coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                7.251e+06   4.04e+06      1.795      0.073   -6.65e+05    1.52e+07
original_language_en[T.True]            -1.578e+06   1.32e+06     -1.192      0.233   -4.17e+06    1.02e+06
release_month_12[T.True]                -1.627e+07   2.25e+06     -7.242      0.000   -2.07e+07   -1.19e+07
popularity                              -1.748e+06   1.32e+05    -13.279      0.000   -2.01e+06   -1.49e+06
popularity:original_language_en[T.True]  1.483e+06    9.5e+04     15.604      0.000     1.3e+06    1.67e+06
popularity:release_month_12[T.True]      2.384e+06    1.3e+05     18.364      0.000    2.13e+06    2.64e+06
runtime                                 -1.218e+05   6.89e+04     -1.768      0.077   -2.57e+05    1.32e+04
Adventure                               -4.971e+07   7.11e+06     -6.996      0.000   -6.36e+07   -3.58e+07
Science_Fiction                          -9.12e+07   8.13e+06    -11.213      0.000   -1.07e+08   -7.53e+07
I(popularity ** 2)                      -1082.8879     22.229    -48.716      0.000   -1126.458   -1039.318
I(runtime ** 2)                           635.3300    308.555      2.059      0.040      30.536    1240.124
popularity:runtime                       1.492e+04    929.399     16.050      0.000    1.31e+04    1.67e+04
popularity:Adventure                     1.121e+06   4.28e+04     26.175      0.000    1.04e+06     1.2e+06
runtime:Adventure                         6.41e+05   6.87e+04      9.336      0.000    5.06e+05    7.76e+05
runtime:Science_Fiction                  1.046e+06   8.23e+04     12.706      0.000    8.84e+05    1.21e+06
==============================================================================
Omnibus:                    36033.204   Durbin-Watson:                   1.994
Prob(Omnibus):                  0.000   Jarque-Bera (JB):        122100807.340
Skew:                          12.768   Prob(JB):                         0.00
Kurtosis:                     384.928   Cond. No.                     8.77e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.77e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""